# Bank Customer Churn Prediction - V1 (XGBoost - Recall Optimized)

This notebook builds a comprehensive machine learning model to predict customer churn for a bank. It is based on a robust classification template, adapted for the specific challenges and features of this dataset.

**Goal:** Predict "which customers are most likely to churn?" (`Exited` = 1), with a primary focus on maximizing **Recall** to ensure we identify as many potential churners as possible.

**Key Features of this Model:**
*   **Model:** Uses XGBoost, a powerful gradient boosting algorithm known for high performance.
*   **Primary Metric:** The model is tuned and evaluated based on **Recall**, which is critical for churn prediction to minimize missed opportunities for retention.
*   **Feature Engineering:** Based on our initial analysis, several new features are created to capture complex customer behaviors:
    *   `TenureToAgeRatio`: To identify long-term, loyal customers.
    *   `BalanceToSalaryRatio`: A measure of a customer's financial health/savings rate.
    *   `PointsPerTenure`: An engagement metric for credit card usage over time.
    *   `IsTransactionalCustomer`: A flag for customers with low balances but active credit cards.
    *   `ComplaintResolved`: An interaction feature to see if resolving a complaint impacts retention.
*   **Imbalance Handling:** The model is tuned using `scale_pos_weight` in XGBoost, a direct and effective method for handling class imbalance.
*   **Rigorous Evaluation:** Includes extensive overfitting analysis using learning curves, validation curves, and detailed classification reports, all focused on the `recall` metric.

## 1. Setup - Import Libraries

In [ ]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, learning_curve, validation_curve, cross_val_predict
from sklearn.preprocessing import LabelEncoder
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import recall_score, accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc, roc_auc_score, make_scorer
import io

# Import Colab specific tools for file handling
from google.colab import files

# Ignore minor warnings for cleaner output
warnings.filterwarnings('ignore')
# Display options for pandas
pd.set_option('display.max_columns', None) # Show all columns

print("Libraries imported successfully.")

### Install Required Libraries (if not already present)

In [ ]:
!pip install -U xgboost
print("xgboost check/installation complete.")

## 2. Load Data

Upload the `Customer-Churn-Records.csv` file.

In [ ]:
print("Please upload Customer-Churn-Records.csv:")
uploaded_file = files.upload()

# Load File into DataFrame
try:
    filename = list(uploaded_file.keys())[0]
    full_df = pd.read_csv(io.BytesIO(uploaded_file[filename]))
    print("Successfully loaded the dataset.")
    print(f"Dataset shape: {full_df.shape}")
    print("\nData Head:")
    display(full_df.head())
except Exception as e:
    print(f"Error loading data: {e}")
    print("Please ensure you uploaded the correct CSV file.")

## 3. Feature Engineering & Preprocessing

This section prepares the data by creating new features based on our initial hypotheses and converting the data into a model-ready format.

### 3.1 Drop Unnecessary Columns & Create New Features

In [ ]:
# --- Drop Identifier Columns ---
print("Dropping identifier columns: RowNumber, CustomerId, Surname...")
full_df = full_df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

# --- Create New Features based on Brainstorming ---
print("Creating new features...")

# Handle potential division by zero by adding a small epsilon (1e-6)
full_df['TenureToAgeRatio'] = full_df['Tenure'] / (full_df['Age'] + 1e-6)
full_df['BalanceToSalaryRatio'] = full_df['Balance'] / (full_df['EstimatedSalary'] + 1e-6)

# Add 1 to Tenure to avoid division by zero for new customers (Tenure=0)
full_df['PointsPerTenure'] = full_df['Point Earned'] / (full_df['Tenure'] + 1)

# Create boolean flags for specific customer personas
full_df['IsTransactionalCustomer'] = ((full_df['Balance'] == 0) & (full_df['HasCrCard'] == 1)).astype(int)
full_df['ComplaintResolved'] = ((full_df['Complain'] == 1) & (full_df['Satisfaction Score'] >= 4)).astype(int)

print("New features created: 'TenureToAgeRatio', 'BalanceToSalaryRatio', 'PointsPerTenure', 'IsTransactionalCustomer', 'ComplaintResolved'")

print("\nDataFrame head after adding new features:")
display(full_df[['Tenure', 'Age', 'TenureToAgeRatio', 'Balance', 'EstimatedSalary', 'BalanceToSalaryRatio', 'Point Earned', 'PointsPerTenure', 'IsTransactionalCustomer', 'ComplaintResolved']].head())

### 3.2 Convert Categorical Features

In [ ]:
# --- One-Hot Encode Nominal Features ---
cols_to_encode = ['Geography', 'Gender', 'Card Type']
print(f"One-Hot Encoding: {cols_to_encode}...")

processed_df = pd.get_dummies(full_df, columns=cols_to_encode, drop_first=True)

print("\nDataFrame head after categorical conversion:")
display(processed_df.head())
print("\nPreprocessing complete. Final features:")
print(processed_df.columns)

## 4. Split Data into Training and Test Sets

In [ ]:
X = processed_df.drop('Exited', axis=1)
y = processed_df['Exited']

# Using Stratified Split to maintain the same proportion of churners in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training features shape: {X_train.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Test features shape: {X_test.shape}")
print(f"Test target shape: {y_test.shape}")

## 5. Check Target Variable Distribution (Imbalance)

In [ ]:
print("Distribution of target variable 'Exited' in y_train:")
churn_distribution = y_train.value_counts(normalize=True)
print(churn_distribution)

plt.figure(figsize=(6,4))
sns.countplot(x=y_train)
plt.title('Distribution of Churn (Exited) in Training Data')
plt.xlabel('Exited (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.show()

# Calculate scale_pos_weight for XGBoost
scale_pos_weight = churn_distribution[0] / churn_distribution[1]
print(f"\nCalculated scale_pos_weight for XGBoost to handle imbalance: {scale_pos_weight:.2f}")

## 6. Hyperparameter Tuning & Model Training (XGBoost)

Using `GridSearchCV` with **`recall`** as the primary scoring metric. We will also include `scale_pos_weight` in our grid search to let the model find the optimal weight for handling the class imbalance.

In [ ]:
print("\nSetting up GridSearchCV for XGBoost (Optimizing for Recall)...")

xgb_model = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')

# A focused parameter grid for V1. Can be expanded for more exhaustive search.
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.7, 0.8],
    'colsample_bytree': [0.7, 0.8],
    'gamma': [0, 0.1],
    # Let's tune scale_pos_weight around the calculated value
    'scale_pos_weight': [scale_pos_weight, scale_pos_weight * 0.8, scale_pos_weight * 1.2]
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(estimator=xgb_model,
                           param_grid=param_grid,
                           cv=cv_strategy,
                           scoring='recall', # PRIMARY METRIC
                           n_jobs=-1, # Use all available cores
                           verbose=2)

print("Starting GridSearchCV... This may take several minutes.")
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

print("\nGridSearchCV Complete.")
print(f"Best Cross-Validation Recall Score found: {grid_search.best_score_:.4f}")
print("Best Parameters found:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

# Evaluate on the full training set using the best model
y_train_pred_best_model = best_model.predict(X_train)
train_recall_best_model = recall_score(y_train, y_train_pred_best_model)
print(f"\nRecall of the best model on the *entire* training set: {train_recall_best_model:.4f}")

print("\nTop 15 Feature Importances from the best XGBoost model:")
importances = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': best_model.feature_importances_
}).sort_values(by='Importance', ascending=False)
print(importances.head(15))

## 7. Overfitting Analysis (Using Best XGBoost & Recall)

Now, we'll analyze the best model for signs of overfitting. Our primary metric of concern is **Recall**.

### 7.1 Training vs. Cross-Validation Recall

In [ ]:
print("\n--- 1. Training vs. Cross-Validation Recall ---")
cv_recall = grid_search.best_score_

print(f"Recall on FULL Training Set (using best model): {train_recall_best_model:.4f}")
print(f"Best Mean Cross-Validation Recall: {cv_recall:.4f}")
print(f"Difference (Train Recall - CV Recall): {train_recall_best_model - cv_recall:.4f}")

if (train_recall_best_model - cv_recall) > 0.10: # Recall can have higher variance, so a slightly larger threshold
    print("\nWarning: Potential overfitting detected! Training Recall is notably higher than CV Recall.")
else:
    print("\nTraining and CV Recall are reasonably close. Lower risk of significant overfitting.")

### 7.2 Learning Curves (Recall)

In [ ]:
print("\n--- 2. Generating Learning Curves (Recall) ---")
train_sizes, train_scores, validation_scores = learning_curve(
    estimator=best_model,
    X=X_train,
    y=y_train,
    train_sizes=np.linspace(0.1, 1.0, 5),
    cv=cv_strategy,
    scoring='recall',
    n_jobs=-1,
    random_state=42
)

train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
validation_scores_mean = np.mean(validation_scores, axis=1)
validation_scores_std = np.std(validation_scores, axis=1)

plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std, alpha=0.1, color="r")
plt.fill_between(train_sizes, validation_scores_mean - validation_scores_std, validation_scores_mean + validation_scores_std, alpha=0.1, color="g")
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training Recall")
plt.plot(train_sizes, validation_scores_mean, 'o-', color="g", label="Cross-validation Recall")
plt.title("Learning Curves (XGBoost - Recall)")
plt.xlabel("Training examples")
plt.ylabel("Recall Score")
plt.legend(loc="best")
plt.grid(True)
plt.ylim(0.0, 1.05)
plt.show()
print("Observe the gap and convergence of training and cross-validation recall curves.")

### 7.3 Classification Reports & Confusion Matrices

In [ ]:
print("\n--- 3. Classification Reports & Confusion Matrices ---")

print("\nClassification Report (Full Training Set - Best Model Predictions):")
print(classification_report(y_train, y_train_pred_best_model, zero_division=0))

try:
    y_train_cv_pred_class = cross_val_predict(best_model, X_train, y_train, cv=cv_strategy, n_jobs=-1)
    print("Generated cross-validated class predictions.")

    print("\nClassification Report (Cross-Validated Predictions on Train Set):")
    print(classification_report(y_train, y_train_cv_pred_class, zero_division=0))

    # Confusion Matrices
    cm_train = confusion_matrix(y_train, y_train_pred_best_model)
    cm_cv = confusion_matrix(y_train, y_train_cv_pred_class)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle('Confusion Matrices Comparison (Best Model)')

    disp_train = ConfusionMatrixDisplay(confusion_matrix=cm_train, display_labels=best_model.classes_)
    disp_train.plot(ax=axes[0], cmap=plt.cm.Blues)
    axes[0].set_title('CM on Full Training Set')

    disp_cv = ConfusionMatrixDisplay(confusion_matrix=cm_cv, display_labels=best_model.classes_)
    disp_cv.plot(ax=axes[1], cmap=plt.cm.Blues)
    axes[1].set_title('CM using Cross-Validated Predictions')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

except Exception as e:
    print(f"Could not generate cross-validated reports or matrices: {e}")

## 8. Final Evaluation on Test Set

This is the final step. We use the `best_model` trained on the full training data to make predictions on the unseen test data (`X_test`). This gives us the most realistic estimate of the model's performance in a real-world scenario.

In [ ]:
print("\nMaking predictions on the held-out test set...")
y_pred_test = best_model.predict(X_test)

print("--- Final Model Evaluation on Test Set ---")
print("Classification Report (Test Set):")
print(classification_report(y_test, y_pred_test, zero_division=0))

print("Confusion Matrix (Test Set):")
cm_test = confusion_matrix(y_test, y_pred_test)
disp_test = ConfusionMatrixDisplay(confusion_matrix=cm_test, display_labels=best_model.classes_)
disp_test.plot(cmap=plt.cm.Greens)
plt.title('Final Confusion Matrix on Unseen Test Data')
plt.show()

final_recall = recall_score(y_test, y_pred_test)
final_accuracy = accuracy_score(y_test, y_pred_test)
print(f"Final Recall on Test Set: {final_recall:.4f}")
print(f"Final Accuracy on Test Set: {final_accuracy:.4f}")

print("\n--- Churn Prediction Process Finished --- ")